# Extracción de Variable Objetivo

Este notebook extrae la variable objetivo (target) del dataset postoperatorio siguiendo una estructura modular.

## Estructura

1. Base de librerías
2. Lectura de datos
3. Funciones generales
4. Funciones específicas de validación por variable/conjunto
5. Pipeline de extracción



## 1. Base de Librerías


In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

# Configuración
pd.set_option('display.max_columns', None)


## 2. Lectura de Datos


In [15]:
# Cargar datasets
df_postqx = pd.read_excel('../../OPERA POSTQX.xlsx')
df_preop_clean = pd.read_csv('../../OPERA_BASE_clean.csv')

print(f"Postoperatorio: {df_postqx.shape[0]:,} filas × {df_postqx.shape[1]} columnas")
print(f"Preoperatorio limpio: {df_preop_clean.shape[0]:,} filas × {df_preop_clean.shape[1]} columnas")


Postoperatorio: 29,865 filas × 76 columnas
Preoperatorio limpio: 30,962 filas × 260 columnas


## 3. Funciones Generales


In [16]:
def has_value(series: pd.Series) -> pd.Series:
    """Retorna True si la serie tiene valor no nulo."""
    return series.notna()


def has_marker(series: pd.Series, marker: str = 'x') -> pd.Series:
    """Retorna True si la serie contiene el marcador especificado."""
    return series.str.lower().str.strip() == marker.lower()


def has_specific_value(series: pd.Series, value: str) -> pd.Series:
    """Retorna True si la serie es igual al valor especificado (case insensitive)."""
    return series.str.upper().str.strip() == value.upper()


def is_not_in_list(series: pd.Series, exclude_list: List[str]) -> pd.Series:
    """Retorna True si el valor NO está en la lista de exclusión."""
    return series.apply(lambda x: pd.notna(x) and x not in exclude_list)

def calculate_duration(start_series: pd.Series, end_series: pd.Series) -> pd.Series:
    """Calcula la duración en minutos entre dos series de tiempo."""
    start = pd.to_datetime(start_series, errors='coerce')
    end = pd.to_datetime(end_series, errors='coerce')
    duration = (end - start).dt.total_seconds() / 60
    return duration


def print_validation_result(name: str, series: pd.Series, total: int) -> None:
    """Imprime el resultado de una validación."""
    n_positive = series.sum()
    print(f"  {name}: {n_positive:,} casos ({n_positive/total*100:.2f}%)")


## 4. Funciones Específicas de Validación

Cada función:
- Recibe un DataFrame
- Crea una copia
- Agrega columna(s) de validación
- Retorna la copia modificada


### Cancelacion del procedimiento

In [119]:
def validate_based_on_cancelacion(df: pd.DataFrame) -> pd.DataFrame:
    """
    Valida basado en cancelación del procedimiento.
    """
    df_result = df.copy()

    df_result['canceladas'] = (df_result['Cirugia'].isna() & df_result['Otros procedimientos'].isna()).astype(int)

    df_result['canceladas_por_medico'] = (df_result['Cancelación de procedimiento - episodio consulta preanes'].str.contains('C-CA por Médico') | df_result['Cancelación de procedimiento - episodio procedimiento'].str.contains('C-CA por Médico')).astype(int)

    df_result['flag_cancelacion'] = (df_result['canceladas'] == 1) & (df_result['canceladas_por_medico'] == 1)

    print("\nVALIDACIÓN CANCELACION:")
    print(f"  Canceladas por medico: {df_result['canceladas_por_medico'].sum():,} ({df_result['canceladas_por_medico'].mean()*100:.2f}%)\n")

    return df_result

### Reservas (Hemoderivados y Sangre)



In [ ]:
def validate_based_on_reservas(df: pd.DataFrame) -> pd.DataFrame:
    """
    Valida basado en reservas de hemoderivados y sangre.
    
    Columnas agregadas:
        - flag_reserva_hemoderivados (código 912002)
        - flag_reserva_sangre (valor = 1)
        - flag_reservas (agregado)
    """
    df_result = df.copy()
    
    df_result['flag_reserva_hemoderivados'] = (df_result['Reserva de hemoderivados'] == 912002).astype(int)
    df_result['flag_reserva_sangre'] = (df_result['Reserva de sangre '] == 1).astype(int)
    df_result['flag_reservas'] = ((df_result['flag_reserva_hemoderivados'] == 1) | (df_result['flag_reserva_sangre'] == 1)).astype(int)
    
    print("VALIDACIÓN RESERVAS:")
    print_validation_result('flag_reserva_hemoderivados', df_result['flag_reserva_hemoderivados'], len(df_result))
    print_validation_result('flag_reserva_sangre', df_result['flag_reserva_sangre'], len(df_result))
    print(f"\n  TOTAL flag_reservas: {df_result['flag_reservas'].sum():,} ({df_result['flag_reservas'].mean()*100:.2f}%)")
    
    return df_result

### Variables Fisiológicas Prequirúrgicas


In [114]:
def validate_based_on_fisiologicas(df: pd.DataFrame) -> pd.DataFrame:
    """
    Valida basado en variables fisiológicas prequirúrgicas con valores anormales.
    Realiza imputación de valores inválidos (nulos, ceros, valores fuera de límites fisiológicos)
    con valores normales antes de evaluar.
    
    Columnas agregadas:
        - flag_presion_sistolica_anormal
        - flag_presion_diastolica_anormal
        - flag_presion_media_anormal
        - flag_frecuencia_cardiaca_anormal
        - flag_saturacion_oxigeno_anormal
        - flag_temperatura_anormal
        - flag_glucometria_anormal
        - flag_fisiologicas (agregado)
    """
    df_result = df.copy()
    
    # Criterios y límites para variables fisiológicas
    criterios_fisiologicas = {
        'Presión arterial sistólica prequirúrgica ': {
            'flag': 'flag_presion_sistolica_anormal',
            'min': 80,
            'max': 150,
            'imputacion': 115,  # Valor normal medio
            'limite_invalido_min': 50,  # Físicamente imposible < 20 mmHg
            'limite_invalido_max': 200  # Físicamente imposible > 300 mmHg
        },
        'Presión arterial diastólica prequirúrgica': {
            'flag': 'flag_presion_diastolica_anormal',
            'min': 50,
            'max': 100,
            'imputacion': 75,  # Valor normal medio
            'limite_invalido_min': 20,  # Físicamente imposible < 10 mmHg
            'limite_invalido_max': 150  # Físicamente imposible > 200 mmHg
        },
        'Presión arterial media prequirúrgica': {
            'flag': 'flag_presion_media_anormal',
            'min': 60,
            'max': 115,
            'imputacion': 87.5,  # Valor normal medio
            'limite_invalido_min': 25,  # Físicamente imposible < 15 mmHg
            'limite_invalido_max': 200  # Físicamente imposible > 250 mmHg
        },
        'Frecuencia cardiaca prequirúrgica': {
            'flag': 'flag_frecuencia_cardiaca_anormal',
            'min': 50,
            'max': 110,
            'imputacion': 80,  # Valor normal medio
            'limite_invalido_min': 25,  # Físicamente imposible < 20 bpm
            'limite_invalido_max': 200  # Físicamente imposible > 250 bpm
        },
        'Saturación de oxígeno prequirúrgica': {
            'flag': 'flag_saturacion_oxigeno_anormal',
            'min': 95,
            'max': None,  # Solo mínimo
            'imputacion': 98,  # Valor normal
            'limite_invalido_min': 50,  # Físicamente imposible < 50%
            'limite_invalido_max': 100  # Máximo posible es 100%
        },
        'Temperatura prequirúrgica': {
            'flag': 'flag_temperatura_anormal',
            'min': 35,
            'max': 37.5,
            'imputacion': 36.75,  # Valor normal medio
            'limite_invalido_min': 30,  # Físicamente imposible < 30°C (hipotermia extrema incompatible con vida)
            'limite_invalido_max': 45  # Físicamente imposible > 45°C (hipertermia extrema incompatible con vida)
        }
    }
    
    # Aplicar imputación y criterios para variables con rango min-max
    for col, criterio in criterios_fisiologicas.items():
        serie = df_result[col].copy()
        
        # Identificar valores inválidos: nulos, ceros, o valores fuera de límites fisiológicos
        invalidos = (
            serie.isna() | 
            (serie == 0) | 
            (serie < criterio['limite_invalido_min']) | 
            (serie > criterio['limite_invalido_max'])
        )
        
        # Imputar valores inválidos con valor normal
        serie_imputada = serie.copy()
        serie_imputada[invalidos] = criterio['imputacion']
        
        # Aplicar criterios de normalidad sobre la serie imputada
        if criterio['max'] is None:
            # Solo mínimo (saturación de oxígeno)
            df_result[criterio['flag']] = (serie_imputada < criterio['min']).astype(int)
        else:
            # Rango min-max
            df_result[criterio['flag']] = ((serie_imputada < criterio['min']) | (serie_imputada > criterio['max'])).astype(int)
    
    # Glucometría: valor = 1 (Sí) o valor fuera de rango normal (70-100 mg/dL)
    glucometria_flag = (df_result['Glucometría prequirúrgica '] == 1).astype(int)
    valor_glucometria = pd.to_numeric(df_result['Valor glucometria '], errors='coerce')
    
    # Imputar valores inválidos de glucometría (nulos, ceros, valores fuera de límites fisiológicos)
    # Límites: < 20 mg/dL (hipoglucemia extrema incompatible con vida) o > 600 mg/dL (hiperglucemia extrema)
    invalidos_glucometria = (
        valor_glucometria.isna() | 
        (valor_glucometria == 0) | 
        (valor_glucometria < 20) | 
        (valor_glucometria > 600)
    )
    valor_glucometria_imputado = valor_glucometria.copy()
    valor_glucometria_imputado[invalidos_glucometria] = 85  # Valor normal medio
    
    glucometria_valor_anormal = ((valor_glucometria_imputado < 70) | (valor_glucometria_imputado > 100)).astype(int)
    df_result['flag_glucometria_anormal'] = ((glucometria_flag == 1) | (glucometria_valor_anormal == 1)).astype(int)
    
    fisiologicas_flags = [
        'flag_presion_sistolica_anormal', 'flag_presion_diastolica_anormal', 'flag_presion_media_anormal',
        'flag_frecuencia_cardiaca_anormal', 'flag_saturacion_oxigeno_anormal', 'flag_temperatura_anormal',
        'flag_glucometria_anormal'
    ]
    df_result['flag_fisiologicas'] = (df_result[fisiologicas_flags].sum(axis=1) > 0).astype(int)
    
    print("\nVALIDACIÓN VARIABLES FISIOLÓGICAS (VALORES ANORMALES):")
    print("  (Valores inválidos imputados con valores normales antes de evaluar)")
    for flag in fisiologicas_flags:
        print_validation_result(flag, df_result[flag], len(df_result))
    print(f"\n  TOTAL flag_fisiologicas: {df_result['flag_fisiologicas'].sum():,} ({df_result['flag_fisiologicas'].mean()*100:.2f}%)")
    
    return df_result

### Tiempos de Procedimiento


In [ ]:
def validate_based_on_tiempos(df: pd.DataFrame) -> pd.DataFrame:
    """
    Valida basado en tiempos de procedimiento con duraciones anormalmente largas.
    
    Columnas agregadas:
        - duracion_cirujano_minutos
        - duracion_anestesia_minutos
        - flag_duracion_cirujano_larga (percentil 90)
        - flag_duracion_anestesia_larga (percentil 90)
        - flag_tiempos (agregado)
    """
    df_result = df.copy()
    
    df_result['duracion_cirujano_minutos'] = calculate_duration(
        df_result['Inicio cirujano '], df_result['Fin cirujano ']
    )
    df_result['duracion_anestesia_minutos'] = calculate_duration(
        df_result['Inicio anestesia '], df_result['Fin anestesia ']
    )
    
    # Duraciones anormalmente largas (percentil 90)
    p90_cirujano = df_result['duracion_cirujano_minutos'].quantile(0.90)
    p90_anestesia = df_result['duracion_anestesia_minutos'].quantile(0.90)
    
    df_result['flag_duracion_cirujano_larga'] = (df_result['duracion_cirujano_minutos'] > p90_cirujano).astype(int)
    df_result['flag_duracion_anestesia_larga'] = (df_result['duracion_anestesia_minutos'] > p90_anestesia).astype(int)
    
    tiempos_flags = ['flag_duracion_cirujano_larga', 'flag_duracion_anestesia_larga']
    df_result['flag_tiempos'] = (df_result[tiempos_flags].sum(axis=1) > 0).astype(int)
    
    print("\nVALIDACIÓN TIEMPOS (DURACIONES LARGAS):")
    print(f"  Percentil 90 cirujano: {p90_cirujano:.1f} minutos")
    print(f"  Percentil 90 anestesia: {p90_anestesia:.1f} minutos")
    for flag in tiempos_flags:
        print_validation_result(flag, df_result[flag], len(df_result))
    print(f"\n  TOTAL flag_tiempos: {df_result['flag_tiempos'].sum():,} ({df_result['flag_tiempos'].mean()*100:.2f}%)")
    
    return df_result

### Clase de Inducción

In [ ]:
def validate_based_on_induccion(df: pd.DataFrame) -> pd.DataFrame:
    """
    Valida basado en clase de inducción (valores que indican mayor complejidad).
    
    Valores posibles: Intravenosa, Ninguna, Inhalatoria, Mixta, o vacío
    
    Columnas agregadas:
        - flag_induccion_compleja (Mixta, Inhalatoria)
        - flag_induccion (agregado)
    """
    df_result = df.copy()
    
    # Normalizar valores (case insensitive, strip)
    clase_induccion = df_result['Clase de inducción '].astype(str).str.strip().str.title()
    
    # Valores que indican mayor complejidad
    valores_complejos = ['Mixta']
    
    df_result['flag_induccion_compleja'] = clase_induccion.isin(valores_complejos).astype(int)
    df_result['flag_induccion'] = df_result['flag_induccion_compleja']
    
    print("\nVALIDACIÓN CLASE DE INDUCCIÓN:")
    print(f"  Valores complejos considerados: {valores_complejos}")
    print_validation_result('flag_induccion_compleja', df_result['flag_induccion_compleja'], len(df_result))
    print(f"\n  TOTAL flag_induccion: {df_result['flag_induccion'].sum():,} ({df_result['flag_induccion'].mean()*100:.2f}%)")
    
    return df_result

### Vía Aérea

In [153]:
def validate_based_on_via_aerea(df: pd.DataFrame) -> pd.DataFrame:
    """
    Valida basado en variables de vía aérea (valores que indican complejidad).
    
    Columnas agregadas:
        - flag_intubacion_dificil (Dificil, Imposible)
        - flag_laringoscopia_alta (valores numéricos altos, percentil 75+)
        - flag_tipo_intubacion_complejo (Nasotraqueal, Otro, Traqueostomia)
        - flag_hoja_laringoscopio_recta (recta)
        - flag_elemento_via_aerea_complejo (Fibrolaringoscopio, Otro)
        - flag_via_aerea (agregado)
    """
    df_result = df.copy()
    
    # Intubación: Ninguna, Facil, Dificil, Imposible
    intubacion = df_result['Intubación  '].astype(str).str.strip().str.title()
    valores_intubacion_dificil = ['Dificil', 'Imposible']
    df_result['flag_intubacion_dificil'] = intubacion.isin(valores_intubacion_dificil).astype(int)
    
    # Laringoscopia: valores numéricos (valores altos indican mayor complejidad)
    laringoscopia = pd.to_numeric(df_result['Laringoscopia '], errors='coerce')
    df_result['flag_laringoscopia_alta'] = (laringoscopia > 1).astype(int)
    
    # Tipo intubación: Orotraqueal, Nasotraqueal, Ninguna, Otro, Traqueostomia
    tipo_intubacion = df_result['Tipo de intubación'].astype(str).str.strip().str.title()
    valores_tipo_complejo = ['Nasotraqueal', 'Otro', 'Traqueostomia']
    df_result['flag_tipo_intubacion_complejo'] = tipo_intubacion.isin(valores_tipo_complejo).astype(int)
    
    # Hoja de laringoscopio: Recta, curva, Ninguna
    hoja_laringoscopio = df_result['Hoja de laringoscopio '].astype(str).str.strip().str.lower()
    df_result['flag_hoja_laringoscopio_recta'] = (hoja_laringoscopio == 'recta').astype(int)
    
    # Elemento vía aérea: Tubo Endotraqueal, Canulanasal, Mascara Laringea, Fibrolaringoscopio, Mascara facial, Ninguno, Otro
    elemento_via_aerea = df_result['Elemento de vía aérea utilizado '].astype(str).str.strip().str.title()
    valores_elemento_complejo = ['Fibrolaringoscopio', 'Otro']
    df_result['flag_elemento_via_aerea_complejo'] = elemento_via_aerea.isin(valores_elemento_complejo).astype(int)
    
    # Agregar flag si hay al menos un indicador de complejidad
    via_aerea_flags = [
        'flag_intubacion_dificil', 'flag_laringoscopia_alta', 'flag_tipo_intubacion_complejo',
        'flag_hoja_laringoscopio_recta', 'flag_elemento_via_aerea_complejo'
    ]
    df_result['flag_via_aerea'] = (df_result[via_aerea_flags].sum(axis=1) > 0).astype(int)
    
    print("\nVALIDACIÓN VÍA AÉREA (COMPLEJIDAD):")
    for flag in via_aerea_flags:
        print_validation_result(flag, df_result[flag], len(df_result))
    print(f"\n  TOTAL flag_via_aerea (≥1 indicador): {df_result['flag_via_aerea'].sum():,} ({df_result['flag_via_aerea'].mean()*100:.2f}%)")
    
    return df_result

### Construcción del Target Final

In [133]:
def build_target(df: pd.DataFrame) -> pd.DataFrame:
    """
    Construye la variable objetivo final combinando todos los flags.

    Columnas agregadas:
        - target
    """
    df_result = df.copy()

    aggregate_flags = [
        'flag_cancelacion',
        'flag_reservas',
        'flag_fisiologicas',
        'flag_tiempos',
        'flag_induccion',
        'flag_via_aerea'
    ]

    # Primero, calcular el target original
    df_result['target'] = (df_result[aggregate_flags].sum(axis=1) > 0).astype(int)

    # Para las filas con canceladas == 1 pero canceladas_por_medico == 0:
    # Siempre debe target = 0, nunca por complicación
    mask_cancelada_no_medico = (df_result['canceladas'] == 1) & (df_result['canceladas_por_medico'] == 0)
    df_result.loc[mask_cancelada_no_medico, 'target'] = 0

    print("\n" + "=" * 70)
    print("VARIABLE OBJETIVO FINAL")
    print("=" * 70)
    print(f"\nFlags utilizados: {aggregate_flags}")
    print(f"\nDistribución del target:")
    print(f"  Clase 0 (No necesitaba valoración): {(df_result['target'] == 0).sum():,} ({(df_result['target'] == 0).mean()*100:.2f}%)")
    print(f"  Clase 1 (Necesitaba valoración): {(df_result['target'] == 1).sum():,} ({(df_result['target'] == 1).mean()*100:.2f}%)")

    return df_result

## 5. Pipeline de Extracción


In [154]:
# ============================================================================
# PIPELINE DE EXTRACCIÓN
# ============================================================================

print("=" * 70)
print("INICIANDO PIPELINE DE EXTRACCIÓN")
print("=" * 70)

df = df_postqx.copy()

df = validate_based_on_cancelacion(df)
df = validate_based_on_reservas(df)
df = validate_based_on_fisiologicas(df)
df = validate_based_on_tiempos(df)
df = validate_based_on_induccion(df)
df = validate_based_on_via_aerea(df)
df = build_target(df)

INICIANDO PIPELINE DE EXTRACCIÓN

VALIDACIÓN CANCELACION:
  Canceladas por medico: 12 (0.04%)

VALIDACIÓN RESERVAS:
  flag_reserva_hemoderivados: 247 casos (0.83%)
  flag_reserva_sangre: 243 casos (0.81%)

  TOTAL flag_reservas: 437 (1.46%)

VALIDACIÓN VARIABLES FISIOLÓGICAS (VALORES ANORMALES):
  (Valores inválidos imputados con valores normales antes de evaluar)
  flag_presion_sistolica_anormal: 1,155 casos (3.87%)
  flag_presion_diastolica_anormal: 684 casos (2.29%)
  flag_presion_media_anormal: 401 casos (1.34%)
  flag_frecuencia_cardiaca_anormal: 913 casos (3.06%)
  flag_saturacion_oxigeno_anormal: 900 casos (3.01%)
  flag_temperatura_anormal: 790 casos (2.65%)
  flag_glucometria_anormal: 980 casos (3.28%)

  TOTAL flag_fisiologicas: 4,716 (15.79%)

VALIDACIÓN TIEMPOS (DURACIONES LARGAS):
  Percentil 90 cirujano: 140.0 minutos
  Percentil 90 anestesia: 168.3 minutos
  flag_duracion_cirujano_larga: 2,279 casos (7.63%)
  flag_duracion_anestesia_larga: 1,597 casos (5.35%)

  TOTAL fl